# RuteStrip Dataset Exploration

Notebook ini memperlihatkan fungsi penting dari dataset **RuteStrip Indonesian Hiking Routes GPX Features and SBERT Embeddings**.

Yang akan dicoba:

- Membaca dataset rute dan embedding.
- Melihat statistik fitur GPX: jarak, elevasi, durasi, grade, dan difficulty.
- Membuat visualisasi sederhana.
- Melakukan pencarian semantik/rekomendasi rute menggunakan cosine similarity antar embedding.
- Membuat baseline klasifikasi difficulty dari fitur numerik.

## 1. Import Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_colwidth', 140)

## 2. Load Dataset

Di Kaggle, dataset biasanya tersedia di folder `/kaggle/input/<dataset-slug>/`. Cell berikut dibuat fleksibel agar bisa berjalan di Kaggle maupun lokal.

In [ ]:
def find_dataset_dir():
    kaggle_input = '/kaggle/input'
    if os.path.exists(kaggle_input):
        for root, dirs, files in os.walk(kaggle_input):
            if 'rutestrip_hiking_routes.csv' in files:
                return root
    return '.'

DATA_DIR = find_dataset_dir()
routes_path = os.path.join(DATA_DIR, 'rutestrip_hiking_routes.csv')
embeddings_path = os.path.join(DATA_DIR, 'rutestrip_hiking_route_embeddings.csv')

routes = pd.read_csv(routes_path)
embeddings = pd.read_csv(embeddings_path)

print('Dataset directory:', DATA_DIR)
print('Routes shape:', routes.shape)
print('Embeddings shape:', embeddings.shape)

In [ ]:
routes.head()

## 3. Ringkasan Dataset

In [ ]:
summary = {
    'total_routes': len(routes),
    'total_mountains': routes['mountain'].nunique(),
    'total_provinces': routes['province'].nunique(),
    'min_distance_km': routes['distance_km'].min(),
    'max_distance_km': routes['distance_km'].max(),
    'avg_distance_km': routes['distance_km'].mean(),
    'avg_elevation_gain_m': routes['elevation_gain_m'].mean(),
    'avg_duration_hour': routes['naismith_duration_hour'].mean(),
}
pd.Series(summary)

In [ ]:
routes['difficulty'].value_counts()

## 4. Visualisasi Fitur GPX

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

routes['distance_km'].hist(ax=axes[0, 0], bins=12)
axes[0, 0].set_title('Distribusi Jarak Rute')
axes[0, 0].set_xlabel('Distance (km)')

routes['elevation_gain_m'].hist(ax=axes[0, 1], bins=12)
axes[0, 1].set_title('Distribusi Elevation Gain')
axes[0, 1].set_xlabel('Elevation Gain (m)')

routes['naismith_duration_hour'].hist(ax=axes[1, 0], bins=12)
axes[1, 0].set_title('Distribusi Estimasi Durasi Naismith')
axes[1, 0].set_xlabel('Duration (hour)')

routes['difficulty'].value_counts().plot(kind='bar', ax=axes[1, 1])
axes[1, 1].set_title('Jumlah Rute per Difficulty')
axes[1, 1].set_xlabel('Difficulty')

plt.tight_layout()

In [ ]:
plt.figure(figsize=(9, 6))
for difficulty, group in routes.groupby('difficulty'):
    plt.scatter(group['distance_km'], group['elevation_gain_m'], label=difficulty, s=70)

plt.title('Distance vs Elevation Gain')
plt.xlabel('Distance (km)')
plt.ylabel('Elevation Gain (m)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 5. Rute Ekstrem dan Rekomendasi Berdasarkan Fitur

Bagian ini menunjukkan contoh insight langsung dari fitur GPX yang sudah diproses.

In [ ]:
cols = ['route_name', 'mountain', 'province', 'distance_km', 'elevation_gain_m', 'naismith_duration_hour', 'average_grade_pct', 'difficulty']

print('Rute terpanjang:')
display(routes.sort_values('distance_km', ascending=False)[cols].head(5))

print('Rute dengan elevation gain tertinggi:')
display(routes.sort_values('elevation_gain_m', ascending=False)[cols].head(5))

print('Rute paling singkat berdasarkan Naismith:')
display(routes.sort_values('naismith_duration_hour')[cols].head(5))

## 6. Semantic Similarity dan Route Recommendation

Dataset menyediakan embedding SBERT untuk tiap rute. Kita bisa memakai cosine similarity untuk mencari rute yang mirip secara semantik berdasarkan `narrative_text`.

In [ ]:
embedding_cols = [col for col in embeddings.columns if col.startswith('embedding_')]
embedding_matrix = embeddings[embedding_cols].to_numpy(dtype=float)

similarity_matrix = cosine_similarity(embedding_matrix)
print('Embedding matrix:', embedding_matrix.shape)
print('Similarity matrix:', similarity_matrix.shape)

In [ ]:
def recommend_similar_routes(route_name_keyword, top_k=5):
    matches = routes[routes['route_name'].str.contains(route_name_keyword, case=False, na=False)]
    if matches.empty:
        raise ValueError(f'No route found for keyword: {route_name_keyword}')

    selected = matches.iloc[0]
    route_id = selected['route_id']
    idx = embeddings.index[embeddings['route_id'] == route_id][0]

    scores = similarity_matrix[idx].copy()
    scores[idx] = -1
    top_indices = np.argsort(scores)[::-1][:top_k]

    result = routes.set_index('route_id').loc[embeddings.iloc[top_indices]['route_id']].reset_index()
    result['similarity_score'] = scores[top_indices]

    print('Selected route:')
    display(selected[cols])
    return result[cols + ['similarity_score']]

recommend_similar_routes('Argopuro', top_k=5)

## 7. Query by Example

Karena dataset ini sudah berisi embedding rute, kita bisa mencari rute yang paling dekat dengan rute contoh. Untuk pencarian teks bebas seperti aplikasi RuteStrip, query perlu diubah dulu menjadi embedding menggunakan model SBERT yang sama.

In [ ]:
example_routes = routes[['route_id', 'route_name', 'narrative_text']].head(3)
example_routes

Jika ingin menjalankan pencarian teks bebas di Kaggle, aktifkan internet dan install/load `sentence-transformers`, lalu encode query dengan model yang sama.

In [ ]:
# Optional: free-text semantic search. Requires sentence-transformers and model availability.
# from sentence_transformers import SentenceTransformer
# model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
# query = 'jalur pendek untuk pemula dengan durasi singkat'
# query_embedding = model.encode([query])
# query_scores = cosine_similarity(query_embedding, embedding_matrix)[0]
# top_indices = np.argsort(query_scores)[::-1][:5]
# routes.iloc[top_indices][cols].assign(similarity_score=query_scores[top_indices])

## 8. Baseline Difficulty Classification

Contoh sederhana: gunakan fitur numerik GPX untuk memprediksi `difficulty`. Karena dataset kecil, hasil ini hanya baseline eksplorasi, bukan model final.

In [ ]:
feature_cols = [
    'distance_km',
    'elevation_gain_m',
    'naismith_duration_hour',
    'average_grade_pct',
    'min_elevation_m',
    'max_elevation_m',
    'total_trackpoints',
]

df_model = routes.dropna(subset=feature_cols + ['difficulty']).copy()
X = df_model[feature_cols]
y = df_model['difficulty']

min_class_count = y.value_counts().min()
n_splits = max(2, min(3, min_class_count))

model = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy')

print('CV folds:', n_splits)
print('Accuracy scores:', scores)
print('Mean accuracy:', scores.mean())

In [ ]:
model.fit(X, y)
importance = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
importance

In [ ]:
importance.sort_values().plot(kind='barh', figsize=(8, 5), title='Feature Importance for Difficulty Classification')
plt.xlabel('Importance')
plt.show()

## 9. Kesimpulan

Dataset ini cocok untuk beberapa eksperimen penting:

- Analisis statistik rute pendakian berbasis GPX.
- Rekomendasi rute mirip menggunakan embedding SBERT.
- Pencarian semantik rute pendakian dalam Bahasa Indonesia.
- Eksperimen klasifikasi difficulty dari fitur numerik.
- Eksplorasi hubungan jarak, elevasi, grade, dan estimasi durasi.